# Lab: Logistic Regression for Text Classification with Binary and Multiclass Datasets

### Introduction

In this lab, you will use logistic regression to classify text in both binary and multiclass settings, gaining hands-on experience with text preprocessing, feature extraction, and logistic regression modeling. You will work with two datasets:
1. **Binary sentiment classification** using the **IMDb Movie Reviews** dataset.
2. **Multiclass sentiment classification** using the **Amazon Fine Food Reviews** dataset.

By the end of this lab, you will:
1. Preprocess text data, including optional steps like stemming and lemmatization.
2. Extract features using TF-IDF, understanding its mathematical basis and seeing a small example.
3. Implement logistic regression for binary and multiclass classification using sklearn.
4. Use cross-validation and evaluate model performance on test sets.
5. Interpret model outputs and evaluate their performance.

---

### Datasets

1. **Binary Classification Dataset (IMDb Movie Reviews)**  
   - **Description:** 50,000 movie reviews labeled as positive or negative.
   - **Classes:** Positive, Negative


2. **Multiclass Classification Dataset (Amazon Fine Food Reviews)**  
   - **Description:** 50,000 food reviews with ratings from 1 to 5.
   - **Classes:** 1-star, 2-star, 3-star, 4-star, 5-star.


In [14]:
!apt-get install unrar
!unrar x datasets.rar
!ls

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'unrar' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.




### Exercise 1: Text Preprocessing and Dataset Exploration

**Goal:** Clean the text data, explore basic statistics, and apply optional text processing techniques.

#### Task:
1. **Load each dataset** and display basic statistics:
   - The number of instances per class.
   - Average number of words per review.
   - Vocabulary size (unique word count).

2. **Text Preprocessing** (for both datasets):
   - Convert text to lowercase.
   - Remove punctuation, numbers, and special characters.
   - Tokenize the text.
   - **Optional Steps**:
     - **Stemming**: Reduce words to their base form (e.g., `running` → `run`).
     - **Lemmatization**: Convert words to their root form using context (e.g., `better` → `good`).

#### Code Hints:
Use `nltk` for stemming and lemmatization (e.g., `PorterStemmer` for stemming and `WordNetLemmatizer` for lemmatization).

3. **Split the dataset** into training and testing sets.
   
4. **Visualize the dataset statistics**. Create plots to show the class distribution, average word count per review, and vocabulary size for each dataset.


In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import string
import re
def preprocess_text(text, stem=False, lemmatize=False):
    """
    Preprocess text data.

    Parameters:
    text (str): Input text to be preprocessed.
    stem (bool): Whether to apply stemming.
    lemmatize (bool): Whether to apply lemmatization.

    Returns:
    str: Preprocessed text.
    """
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    if stem:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(word) for word in tokens]
    
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)


def load_and_preprocess_dataset(dataset_path, preprocess=False, stem=False, lemmatize=False):
    """
    Load and preprocess a dataset.

    Parameters:
    dataset_path (str): Path to the CSV file containing the dataset.
    preprocess (bool): Whether to preprocess the text data.
    stem (bool): Whether to apply stemming during preprocessing.
    lemmatize (bool): Whether to apply lemmatization during preprocessing.

    Returns:
    pandas.DataFrame: Preprocessed dataset.
    """
    # Load the dataset
    if 'Amazon_Food_Reviews.csv' in dataset_path:
        dataset = pd.read_csv(dataset_path, usecols=['Text', 'Score'])
        dataset = dataset.rename(columns={'Text': 'text', 'Score': 'label'})
        # Keep the original score as the label
    elif 'IMDB_Dataset.csv' in dataset_path:
        dataset = pd.read_csv(dataset_path, usecols=['review', 'sentiment'])
        dataset = dataset.rename(columns={'review': 'text', 'sentiment': 'label'})
        dataset['label'] = dataset['label'].apply(lambda x: 1 if x == 'positive' else 0)
    else:
        raise ValueError("Invalid dataset path. Please provide the correct path.")

    # Preprocess the text data (if specified)
    if preprocess:
        dataset['text'] = dataset['text'].apply(lambda x: preprocess_text(x, stem, lemmatize))

    return dataset

def split_data(dataset, test_size=0.2, random_state=42):
    """
    Split the dataset into training and testing sets.

    Parameters:
    dataset (pandas.DataFrame): The dataset to split.
    test_size (float): Proportion of the dataset to include in the test split.
    random_state (int): Random state for reproducibility.

    Returns:
    tuple: (X_train, X_test, y_train, y_test)
    """
    X_train, X_test, y_train, y_test = train_test_split(
        dataset['text'], dataset['label'], test_size=test_size, random_state=random_state
    )
    return X_train, X_test, y_train, y_test

def display_statistics(dataset, dataset_name):
    """
    Display statistics for a dataset.

    Parameters:
    dataset (pandas.DataFrame): The dataset to analyze.
    dataset_name (str): Name of the dataset for labeling purposes.
    """
    print(f"\n=== {dataset_name} Dataset Statistics ===")
    print(f"Total instances: {len(dataset)}")
    print(f"\nClass distribution:")
    print(dataset['label'].value_counts().sort_index())
    
    dataset['word_count'] = dataset['text'].apply(lambda x: len(str(x).split()))
    print(f"\nAverage words per review: {dataset['word_count'].mean():.2f}")
    
    all_words = ' '.join(dataset['text'].astype(str)).split()
    vocab_size = len(set(all_words))
    print(f"Vocabulary size: {vocab_size}")


# Example usage
amazon_df = load_and_preprocess_dataset('Amazon_Food_Reviews.csv', preprocess=True, stem=False, lemmatize=True)
X_train_amazon, X_test_amazon, y_train_amazon, y_test_amazon = split_data(amazon_df)
display_statistics(amazon_df, "Amazon")

imdb_df = load_and_preprocess_dataset('IMDB_Dataset.csv', preprocess=True, stem=False, lemmatize=True)
X_train_imdb, X_test_imdb, y_train_imdb, y_test_imdb = split_data(imdb_df)
display_statistics(imdb_df, "IMDB")



=== Amazon Dataset Statistics ===
Total instances: 50000

Class distribution:
label
1    10000
2    10000
3    10000
4    10000
5    10000
Name: count, dtype: int64

Average words per review: 44.54
Vocabulary size: 39438

=== IMDB Dataset Statistics ===
Total instances: 50000

Class distribution:
label
0    25000
1    25000
Name: count, dtype: int64

Average words per review: 122.22
Vocabulary size: 91499


In [16]:
# Print the first 5 rows and statistics for both datasets
print("Amazon Food Reviews - First 5 Rows and Labels:")
for text, label in zip(X_train_amazon.head(5), y_train_amazon.head(5)):
    print(f"\nText: {text} \nLabel: {label}")

print("\n\n---------------------------------------------------------------------\n\n")

print("\nIMDB Dataset - First 5 Rows and Labels:")
for text, label in zip(X_train_imdb.head(5), y_train_imdb.head(5)):
    print(f"\nText: {text} \nLabel: {label}")

Amazon Food Reviews - First 5 Rows and Labels:

Text: like br low calorie snack also good br taste like dried granny smith apple br bad price getting dozen bag br long shelf life well br added sugar apple br br prefer fuji apple sweeter comparison br granny smith sour 
Label: 4

Text: get local supermarket less price buying one canister time discount buying two time actually checking amazon hoping better price getting giant br br love tangy flavor especially like freshen spike flavor little squeezing half lemon oz glass plenty crushed ice br br pretty good tasting sweetener still convinced aspartame harmless think people sensitive may one especially since like flavor using third powder suggested 
Label: 4

Text: bone last dog month holding well bone hol dog love chew bone seem wear much buy bone 
Label: 5

Text: kick rump worked week past amazon return stop working coarse amazon anyway call ask pay ship back exchange another one livestock hot wire keep long get one back interesting one

### Exercise 2: Feature Extraction with TF-IDF

**Goal:** Represent text numerically using TF-IDF and understand how TF-IDF scoring works.

**TF-IDF Explanation**:
TF-IDF stands for **Term Frequency-Inverse Document Frequency**. It highlights words that are important in specific documents but less common across the entire dataset. This score is computed as follows:

- **Term Frequency (TF)**: Measures the frequency of a term in a document.
  $$ \text{TF}(w, d) = \frac{\text{count of } w \text{ in } d}{\text{total words in } d} $$

- **Inverse Document Frequency (IDF)**: Measures how unique a term is across all documents.
  $$ \text{IDF}(w) = \log \frac{N}{\text{DF}(w)} $$
  where $ N $ is the total number of documents, and $ \text{DF}(w) $ is the document frequency of $ w $.

Thus:
$$ \text{TF-IDF}(w, d) = \text{TF}(w, d) \times \text{IDF}(w) $$


We will use `sklearn.feature_extraction.text.TfidfVectorizer` to calculate TF-IDF.

#### Code Explanation:
- **`TfidfVectorizer`**:
  - **Input**: Raw text data.
  - **Output**: Sparse matrix of TF-IDF scores for each word.
  - **Parameters**: `max_features` (limits vocabulary size), `ngram_range` (for n-grams), `stop_words` (removes common words).

#### Task:
1. **TF-IDF Calculation**: Use sklearn TF-IDF vectorization on each dataset.
2. Print feature values for a few reviews.
3. What role do the parameters `max_features` and `ngram_range` play in the TF-IDF vectorization?

4. **New Document Transformation**:  
   Use the fitted `TfidfVectorizer` to transform a new given document into a TF-IDF vector.


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_tfidf_features(X_train, X_test, max_features=5000, ngram_range=(1,1), sublinear_tf=False):
    """
    Extract TF-IDF features from training and test text data.

    Parameters:
    X_train (pd.Series): Training text data.
    X_test (pd.Series): Test text data.
    max_features (int): Maximum number of features to keep in the vocabulary. Default is 5000.
    ngram_range (tuple): The lower and upper boundary of the range of n-values for n-grams. Default is (1,1) for unigrams.
    sublinear_tf (bool): Apply sublinear term frequency scaling if True. Default is False.

    Returns:
    tuple: (train_tfidf, test_tfidf, tfidf)
        - train_tfidf (sparse matrix): TF-IDF feature matrix for the training data.
        - test_tfidf (sparse matrix): TF-IDF feature matrix for the test data.
        - tfidf (TfidfVectorizer): Fitted TfidfVectorizer object containing vocabulary and IDF values.

    Example:
    >>> X_train = pd.Series(["This is a sample sentence.", "Another example sentence."])
    >>> X_test = pd.Series(["This is a test sentence."])
    >>> train_tfidf, test_tfidf, tfidf = get_tfidf_features(X_train, X_test)
    >>> train_tfidf.shape
    (2, 5)  # Sparse matrix with 2 documents and 5 unique features
    """
    tfidf = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        sublinear_tf=sublinear_tf,
        stop_words='english'
    )
    
    train_tfidf = tfidf.fit_transform(X_train)
    test_tfidf = tfidf.transform(X_test)
    
    return train_tfidf, test_tfidf, tfidf
def print_tfidf_vector(sent, sent_tfidf_vector, feature_names):
    """
    Print TF-IDF values for each word in the specified sentence.

    Parameters:
    sent (str): The sentence for which TF-IDF values are to be printed.
    sent_tfidf_vector (numpy.ndarray): The TF-IDF vector representation of the sentence (dense array).
    feature_names (list): List of feature names corresponding to TF-IDF features in the vectorizer.

    Example:
    >>> sent = "This is a sample sentence."
    >>> sent_tfidf_vector = array([[0.0, 0.5, 0.0, 0.7, 0.0]])
    >>> feature_names = ["sample", "sentence", "test", "this", "is"]
    >>> print_tfidf_vector(sent, sent_tfidf_vector, feature_names)

    Output:
    Words in sentence:
    ['This', 'is', 'a', 'sample', 'sentence.']
    Sorted words by TF-IDF score (highest to lowest) and vocabulary presence:
    Word: sample, TF-IDF: 0.5000, In Vocab: True
    Word: sentence, TF-IDF: 0.7000, In Vocab: True
    Word: this, TF-IDF: 0.0000, In Vocab: True
    Word: is, TF-IDF: 0.0000, In Vocab: True
    """
    print(f"\nWords in sentence:")
    print(sent.split()[:10])
    
    sent_tfidf_dense = sent_tfidf_vector.flatten()
    word_scores = [(feature_names[i], sent_tfidf_dense[i]) 
                   for i in range(len(sent_tfidf_dense)) if sent_tfidf_dense[i] > 0]
    word_scores.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\nTop 10 words by TF-IDF score:")
    for word, score in word_scores[:10]:
        print(f"Word: {word}, TF-IDF: {score:.4f}")


def transform_new_document(new_document, tfidf_vectorizer):
    """
    Transform a new document into a TF-IDF vector using a fitted TfidfVectorizer.

    Parameters:
    new_document (str): The new document (text) to be transformed into TF-IDF vector form.
    tfidf_vectorizer (TfidfVectorizer): A fitted TfidfVectorizer object containing the vocabulary and IDF values.

    Returns:
    numpy.ndarray: The TF-IDF vector (dense format) representation of the new document.

    Example:
    >>> new_document = "This is a new sentence for testing."
    >>> new_document_tfidf_vector = transform_new_document(new_document, tfidf)
    >>> new_document_tfidf_vector.shape
    (1, 5)  # Dense array with 1 document and 5 features
    """
    new_doc_tfidf = tfidf_vectorizer.transform([new_document])
    return new_doc_tfidf.toarray()




# Example usage with amazon dataset
train_tfidf_amazon, test_tfidf_amazon, tfidf_amazon = get_tfidf_features(X_train_amazon, X_test_amazon)
train_tfidf_imdb, test_tfidf_imdb, tfidf_imdb = get_tfidf_features(X_train_imdb, X_test_imdb)

# Get the feature names (vocab words) corresponding to the TF-IDF features for amazon
feature_names_amazon = tfidf_amazon.get_feature_names_out()
print(feature_names_amazon[:100])

# Select the first sentence from the training set
train_sentence = X_train_amazon.iloc[0]
print_tfidf_vector(train_sentence, train_tfidf_amazon[0].toarray(), feature_names_amazon)

# New document to transform
new_sent = "I absolutely love this movie! It was amazing and exciting."
new_sent_tfidf_vector = transform_new_document(new_sent, tfidf_amazon)
print("TF-IDF vector for the new sentence:")
print_tfidf_vector(new_sent, new_sent_tfidf_vector, feature_names_amazon)


['ability' 'able' 'absinthe' 'absolute' 'absolutely' 'absorb' 'absorbed'
 'acai' 'accept' 'acceptable' 'accepted' 'access' 'accident'
 'accidentally' 'according' 'accordingly' 'account' 'accurate'
 'accustomed' 'acerola' 'acesulfame' 'ache' 'achieve' 'acid' 'acidic'
 'acidity' 'acne' 'acquired' 'acrid' 'act' 'action' 'active' 'activity'
 'actual' 'actually' 'acv' 'ad' 'add' 'added' 'addict' 'addicted'
 'addicting' 'addiction' 'addictive' 'adding' 'addition' 'additional'
 'additionally' 'additive' 'address' 'adequate' 'adjust' 'admit'
 'admittedly' 'adopted' 'adore' 'adult' 'advance' 'advantage' 'adventure'
 'adventurous' 'adverse' 'advertise' 'advertised' 'advertisement'
 'advertises' 'advertising' 'advertized' 'advice' 'advise' 'advised'
 'aerator' 'aerogarden' 'affect' 'affected' 'afford' 'affordable' 'afraid'
 'african' 'afternoon' 'aftertaste' 'afterward' 'agave' 'age' 'aged'
 'agent' 'aggressive' 'aging' 'ago' 'agree' 'agreed' 'ahead' 'ahoy' 'aid'
 'air' 'airtight' 'airy' 'aisle' 

### Exercise 3: Logistic Regression with Detailed Equations

**Goal:** Implement logistic regression for binary and multiclass cases, understanding each step and the underlying equations.

#### Logistic Regression for Binary Classification

For binary classification, logistic regression models the probability that a given input $x$ belongs to class 1. The model predicts:
$$ P(y = 1 | x) = \sigma(w^T x + b) $$
where:
- $w$ is the weight vector,
- $x$ is the feature vector (TF-IDF representation),
- $b$ is the bias term,
- $ \sigma(z) = \frac{1}{1 + e^{-z}} $ is the sigmoid function.

The **cross-entropy loss** for binary logistic regression is:
$$ L(y, \hat{y}) = -\left( y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right) $$
where $ \hat{y} = P(y = 1 | x) $.

#### Logistic Regression for Multiclass Classification

For multiclass, logistic regression uses the **softmax function**:
$$ P(y = k | x) = \frac{e^{w_k^T x + b_k}}{\sum_{j=1}^{K} e^{w_j^T x + b_j}} $$
where $K$ is the number of classes (4 for Amazon reviews).

The **cross-entropy loss** for multiclass is:
$$ L(y, \hat{y}) = -\sum_{k=1}^{K} y_k \log(\hat{y}_k) $$
where $y_k$ is a binary indicator for the true class.

#### Code Explanation:
- **`LogisticRegression`**:
  - For binary: `LogisticRegression(solver='liblinear')`
  - For multiclass: `LogisticRegression(multi_class='multinomial', solver='lbfgs')`
  - **Input**: TF-IDF features, labels.
  - **Output**: Predicted probabilities.

#### Task:
1. Train sklearn logistic regression model on the IMDb dataset (binary).
2. Train sklearn logistic regression model on the Amazon dataset (multiclass).
3. Report precision,recall,fscore results for each dataset.


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def binary_logistic_regression(X_train, y_train, X_test, y_test):
    """
    Implement binary logistic regression.

    Parameters:
    X_train (sparse matrix): Training features.
    y_train (np.ndarray): Training labels.
    X_test (sparse matrix): Test features.
    y_test (np.ndarray): Test labels.

    Returns:
    tuple: (model, y_pred, y_prob)
        model (LogisticRegression): Trained logistic regression model.
        y_pred (np.ndarray): Predicted labels on the test set.
        y_prob (np.ndarray): Predicted probabilities on the test set.
    """
    model = LogisticRegression(solver='liblinear', max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    
    return model, y_pred, y_prob


def multiclass_logistic_regression(X_train, y_train, X_test, y_test):
    """
    Implement multiclass logistic regression.

    Parameters:
    X_train (sparse matrix): Training features.
    y_train (np.ndarray): Training labels.
    X_test (sparse matrix): Test features.
    y_test (np.ndarray): Test labels.

    Returns:
    tuple: (model, y_pred, y_prob)
        model (LogisticRegression): Trained logistic regression model.
        y_pred (np.ndarray): Predicted labels on the test set.
        y_prob (np.ndarray): Predicted probabilities on the test set.
    """
    model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    
    return model, y_pred, y_prob

# Example usage
# Assuming 'train_tfidf', 'test_tfidf', 'y_train', 'y_test' are available

# # Train and evaluate binary logistic regression
binary_model, binary_y_pred, binary_y_prob = binary_logistic_regression(train_tfidf_imdb, y_train_imdb, test_tfidf_imdb, y_test_imdb)

# # Train and evaluate multiclass logistic regression
multiclass_model, multiclass_y_pred, multiclass_y_prob = multiclass_logistic_regression(train_tfidf_amazon, y_train_amazon, test_tfidf_amazon, y_test_amazon)


# # Evaluation metrics for binary logistic regression
print("\nBinary Logistic Regression Evaluation Metrics:")
print(f"Accuracy: {accuracy_score(y_test_imdb, binary_y_pred):.4f}")


# # Evaluation metrics for multiclass logistic regression
print("\nMulticlass Logistic Regression Evaluation Metrics (macro):")
print(f"Accuracy: {accuracy_score(y_test_amazon, multiclass_y_pred):.4f}")



c:\Users\Rosey\anaconda3\envs\NLP\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Binary Logistic Regression Evaluation Metrics:
Accuracy: 0.8829

Multiclass Logistic Regression Evaluation Metrics (macro):
Accuracy: 0.4832


#### Using Train and Test Sets

1. Train models on training data, predict on test data.
2. Compute metrics: accuracy, precision, recall, F1-score.

#### Cross-Validation

1. Perform 5-fold CV with `cross_val_score` for both models.
2. Record mean and standard deviation of CV scores and report accuracy, precision, recall, F1-score for each CV.

In [19]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split


def evaluate_model(X_train, y_train, X_test, y_test, num_classes=2):
    """
    Train a logistic regression model and evaluate it on the test set.

    Parameters:
    X_train, X_test: Training and testing feature sets.
    y_train, y_test: Training and testing labels.
    num_classes (int): Number of classes (2 for binary, >2 for multiclass).

    Returns:
    None: Prints the classification report and accuracy metrics.
    """
    # Initialize the logistic regression model based on binary or multiclass classification
    if num_classes == 2:
        model = LogisticRegression(solver='liblinear', max_iter=1000)
        print("\n=== Binary Logistic Regression on Test Set ===")
    else:
        model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
        print("\n=== Multiclass Logistic Regression on Test Set ===")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    
    if num_classes == 2:
        print(f"Precision: {precision_score(y_test, y_pred):.4f}")
        print(f"Recall: {recall_score(y_test, y_pred):.4f}")
        print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    else:
        print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
        print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
        print(f"F1-Score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))



def evaluate_cross_validation(X, y, num_classes=2, num_folds=5):
    """
    Perform cross-validation on the logistic regression model with detailed logging.

    Parameters:
    X (sparse matrix or ndarray): Features.
    y (np.ndarray): Labels.
    num_classes (int): Number of classes (2 for binary, more than 2 for multiclass).
    num_folds (int): Number of cross-validation folds.

    Returns:
    list: Cross-validation accuracy scores for each fold.
    """

    if num_classes == 2:
        model = LogisticRegression(solver='liblinear', max_iter=1000)
        print(f"\n=== Binary Logistic Regression - {num_folds}-Fold Cross-Validation ===")
    else:
        model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
        print(f"\n=== Multiclass Logistic Regression - {num_folds}-Fold Cross-Validation ===")
    
    accuracy_scores = cross_val_score(model, X, y, cv=num_folds, scoring='accuracy')
    precision_scores = cross_val_score(model, X, y, cv=num_folds, scoring='precision_macro')
    recall_scores = cross_val_score(model, X, y, cv=num_folds, scoring='recall_macro')
    f1_scores = cross_val_score(model, X, y, cv=num_folds, scoring='f1_macro')
    
    print(f"\nAccuracy: {accuracy_scores.mean():.4f} (+/- {accuracy_scores.std():.4f})")
    print(f"Precision: {precision_scores.mean():.4f} (+/- {precision_scores.std():.4f})")
    print(f"Recall: {recall_scores.mean():.4f} (+/- {recall_scores.std():.4f})")
    print(f"F1-Score: {f1_scores.mean():.4f} (+/- {f1_scores.std():.4f})")
    
    print(f"\nScores per fold:")
    for i in range(num_folds):
        print(f"Fold {i+1}: Accuracy={accuracy_scores[i]:.4f}, "
              f"Precision={precision_scores[i]:.4f}, "
              f"Recall={recall_scores[i]:.4f}, "
              f"F1={f1_scores[i]:.4f}")
    
    return accuracy_scores

    return None


# # Evaluate binary logistic regression on IMDB dataset
evaluate_model(train_tfidf_imdb, y_train_imdb, test_tfidf_imdb, y_test_imdb, num_classes=2)

# # Evaluate multiclass logistic regression on Amazon dataset
evaluate_model(train_tfidf_amazon, y_train_amazon, test_tfidf_amazon, y_test_amazon, num_classes=3)


# # Example usage for binary classification
evaluate_cross_validation(train_tfidf_imdb, y_train_imdb, num_classes=2)

# # Example usage for multiclass classification
evaluate_cross_validation(train_tfidf_amazon, y_train_amazon, num_classes=4)



=== Binary Logistic Regression on Test Set ===
Accuracy: 0.8829
Precision: 0.8741
Recall: 0.8968
F1-Score: 0.8853

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      4961
           1       0.87      0.90      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


=== Multiclass Logistic Regression on Test Set ===


c:\Users\Rosey\anaconda3\envs\NLP\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.4832
Precision (macro): 0.4771
Recall (macro): 0.4835
F1-Score (macro): 0.4796

Classification Report:
              precision    recall  f1-score   support

           1       0.58      0.62      0.60      2043
           2       0.41      0.38      0.39      1993
           3       0.39      0.35      0.37      2016
           4       0.44      0.43      0.43      1998
           5       0.57      0.63      0.60      1950

    accuracy                           0.48     10000
   macro avg       0.48      0.48      0.48     10000
weighted avg       0.48      0.48      0.48     10000


=== Binary Logistic Regression - 5-Fold Cross-Validation ===

Accuracy: 0.8821 (+/- 0.0024)
Precision: 0.8823 (+/- 0.0024)
Recall: 0.8821 (+/- 0.0024)
F1-Score: 0.8821 (+/- 0.0024)

Scores per fold:
Fold 1: Accuracy=0.8861, Precision=0.8864, Recall=0.8862, F1=0.8861
Fold 2: Accuracy=0.8806, Precision=0.8808, Recall=0.8806, F1=0.8806
Fold 3: Accuracy=0.8832, Precision=0.8834, Recall=0.8833, F1

c:\Users\Rosey\anaconda3\envs\NLP\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Rosey\anaconda3\envs\NLP\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Rosey\anaconda3\envs\NLP\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Rosey\anaconda3\envs\NLP\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'm


Accuracy: 0.4792 (+/- 0.0051)
Precision: 0.4726 (+/- 0.0055)
Recall: 0.4791 (+/- 0.0051)
F1-Score: 0.4750 (+/- 0.0053)

Scores per fold:
Fold 1: Accuracy=0.4736, Precision=0.4675, Recall=0.4735, F1=0.4698
Fold 2: Accuracy=0.4876, Precision=0.4819, Recall=0.4876, F1=0.4840
Fold 3: Accuracy=0.4800, Precision=0.4735, Recall=0.4799, F1=0.4759
Fold 4: Accuracy=0.4804, Precision=0.4734, Recall=0.4803, F1=0.4759
Fold 5: Accuracy=0.4741, Precision=0.4665, Recall=0.4740, F1=0.4693


array([0.473625, 0.487625, 0.48    , 0.480375, 0.474125])

## A detailed example illustrating how tf-idf works

In this improved example, we'll use a small corpus consisting of four words: **"apple"**, **"banana"**, **"fruit"**, and **"eat"**. We will adjust the word frequencies across sentences to ensure that the TF-IDF values for words differ more significantly, so that some words have higher importance than others. To achieve this, we will duplicate certain words in the sentences to manipulate their frequency.

### Step 1: Preprocess and Tokenize the Corpus

The vocabulary we are using is: `["apple", "banana", "fruit", "eat"]`.

#### Sentences:
1. **Sentence 1**: "I eat apple apple"
2. **Sentence 2**: "Banana fruit fruit fruit"
3. **Sentence 3**: "Eat fruit apple"
4. **Sentence 4**: "Banana apple eat eat"

### Step 2: Construct the Term Frequency (TF)

The Term Frequency (TF) for each word is calculated as the number of times a word appears in a sentence divided by the total number of words in that sentence.

**Term Frequency (TF)**: Measures the frequency of a term in a document.
  $$ \text{TF}(w, d) = \frac{\text{count of } w \text{ in } d}{\text{total words in } d} $$

#### Sentence 1: "I eat apple apple"
- Total words: 4
- Word occurrences:
  - "apple" occurs 2 times
  - "eat" occurs 1 time
  - "banana" and "fruit" are absent

TF values for Sentence 1:
- $TF(\text{apple}) = \frac{2}{4} = 0.5$
- $TF(\text{eat}) = \frac{1}{4} = 0.25$
- $TF(\text{banana}) = 0$
- $TF(\text{fruit}) = 0$

#### Sentence 2: "Banana fruit fruit fruit"
- Total words: 4
- Word occurrences:
  - "banana" occurs 1 time
  - "fruit" occurs 3 times

TF values for Sentence 2:
- $TF(\text{apple}) = 0$
- $TF(\text{eat}) = 0$
- $TF(\text{banana}) = \frac{1}{4} = 0.25$
- $TF(\text{fruit}) = \frac{3}{4} = 0.75$

#### Sentence 3: "Eat fruit apple"
- Total words: 3
- Word occurrences:
  - "apple" occurs 1 time
  - "fruit" occurs 1 time
  - "eat" occurs 1 time

TF values for Sentence 3:
- $TF(\text{apple}) = \frac{1}{3} \approx 0.333$
- $TF(\text{eat}) = \frac{1}{3} \approx 0.333$
- $TF(\text{banana}) = 0$
- $TF(\text{fruit}) = \frac{1}{3} \approx 0.333$

#### Sentence 4: "Banana apple eat eat"
- Total words: 4
- Word occurrences:
  - "apple" occurs 1 time
  - "banana" occurs 1 time
  - "eat" occurs 2 times

TF values for Sentence 4:
- $TF(\text{apple}) = \frac{1}{4} = 0.25$
- $TF(\text{eat}) = \frac{2}{4} = 0.5$
- $TF(\text{banana}) = \frac{1}{4} = 0.25$
- $TF(\text{fruit}) = 0$

### Step 3: Calculate the Inverse Document Frequency (IDF)

The Inverse Document Frequency (IDF) is calculated using the following formula:

$$
IDF(w) = \log\left(\frac{N}{df(w)}\right)
$$

where:
- $N$ is the total number of documents (sentences).
- $df(w)$ is the number of sentences in which the word $w$ appears.

#### Calculate the IDF for each word:

1. **apple** appears in 3 sentences (Sentence 1, Sentence 3, Sentence 4).
   - $df(\text{apple}) = 3$
   - $IDF(\text{apple}) = \log\left(\frac{4}{3}\right) \approx 0.288$

2. **banana** appears in 2 sentences (Sentence 2, Sentence 4).
   - $df(\text{banana}) = 2$
   - $IDF(\text{banana}) = \log\left(\frac{4}{2}\right) = \log(2) \approx 0.693$

3. **fruit** appears in 2 sentences (Sentence 2, Sentence 3).
   - $df(\text{fruit}) = 2$
   - $IDF(\text{fruit}) = \log\left(\frac{4}{2}\right) \approx 0.693$

4. **eat** appears in 3 sentences (Sentence 1, Sentence 3, Sentence 4).
   - $df(\text{eat}) = 3$
   - $IDF(\text{eat}) = \log\left(\frac{4}{3}\right) \approx 0.288$

### Step 4: Calculate the TF-IDF

Now, we calculate the TF-IDF for each word in each sentence using the formula:

$$
TF\text{-}IDF(w, d) = TF(w, d) \times IDF(w)
$$

#### Sentence 1: "I eat apple apple"

- $TF\text{-}IDF(\text{apple}, \text{Sentence 1}) = 0.5 \times 0.288 \approx 0.144$
- $TF\text{-}IDF(\text{eat}, \text{Sentence 1}) = 0.25 \times 0.288 \approx 0.072$
- $TF\text{-}IDF(\text{banana}, \text{Sentence 1}) = 0$
- $TF\text{-}IDF(\text{fruit}, \text{Sentence 1}) = 0$

#### Sentence 2: "Banana fruit fruit fruit"

- $TF\text{-}IDF(\text{apple}, \text{Sentence 2}) = 0$
- $TF\text{-}IDF(\text{eat}, \text{Sentence 2}) = 0$
- $TF\text{-}IDF(\text{banana}, \text{Sentence 2}) = 0.25 \times 0.693 \approx 0.173$
- $TF\text{-}IDF(\text{fruit}, \text{Sentence 2}) = 0.75 \times 0.693 \approx 0.520$

#### Sentence 3: "Eat fruit apple"

- $TF\text{-}IDF(\text{apple}, \text{Sentence 3}) = 0.333 \times 0.288 \approx 0.096$
- $TF\text{-}IDF(\text{eat}, \text{Sentence 3}) = 0.333 \times 0.288 \approx 0.096$
- $TF\text{-}IDF(\text{banana}, \text{Sentence 3}) = 0$
- $TF\text{-}IDF(\text{fruit}, \text{Sentence 3}) = 0.333 \times 0.693 \approx 0.231$

#### Sentence 4: "Banana apple eat eat"

- $TF\text{-}IDF(\text{apple}, \text{Sentence 4}) = 0.25 \times 0.288 \approx 0.072$
- $TF\text{-}IDF(\text{eat}, \text{Sentence 4}) = 0.5 \times 0.288 \approx 0.144$
- $TF\text{-}IDF(\text{banana}, \text{Sentence 4}) = 0.25 \times 0.693 \approx 0.173$
- $TF\text{-}IDF(\text{fruit}, \text{Sentence 4}) = 0$

### Step 5: Summary of Results

Here is the summary of the TF-IDF values for each word in the sentences:

| Word   | Sentence 1 | Sentence 2 | Sentence 3 | Sentence 4 |
|--------|------------|------------|------------|------------|
| apple  | 0.144      | 0          | 0.096      | 0.072      |
| banana | 0          | 0.173      | 0          | 0.173      |
| fruit  | 0          | 0.520      | 0.231      | 0          |
| eat    | 0.072      | 0          | 0.096      | 0.144      |

### Step 6: Explanation of the TF-IDF Distribution

- **apple** has moderate importance in sentences where it appears, with highest weight in Sentence 1 due to its double occurrence.
- **banana** has consistent higher importance in both sentences where it appears due to its rarity (appears in only 2 sentences).
- **fruit** has the highest importance in Sentence 2 due to both its relative rarity (2 sentences) and high frequency within that sentence (3 occurrences).
- **eat** shows moderate importance across its appearances, with highest weight in Sentence 4 due to its double occurrence.
